<a href="https://colab.research.google.com/github/sohailarizk/rainfall-prediction-project/blob/main/Classification_and_Captioning_Aircraft_Damage_Using_Pretrained_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
!pip install pandas==2.2.3
!pip install tensorflow_cpu==2.17.1
!pip install pillow==11.1.0
!pip install matplotlib==3.9.2
!pip install transformers==4.38.2
!pip install torch==2.2.0+cpu torchvision==0.17.0+cpu torchaudio==2.2.0+cpu \
    --index-url https://download.pytorch.org/whl/cpu

  Using cached ml_dtypes-0.4.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
  Using cached protobuf-4.25.9-cp37-abi3-manylinux2014_x86_64.whl.metadata (541 bytes)
  Using cached tensorboard-2.17.1-py3-none-any.whl.metadata (1.6 kB)
Using cached ml_dtypes-0.4.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.2 MB)
Using cached protobuf-4.25.9-cp37-abi3-manylinux2014_x86_64.whl (295 kB)
Using cached tensorboard-2.17.1-py3-none-any.whl (5.5 MB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.35.1
    Uninstalling protobuf-7.35.1:
      Successfully uninstalled protobuf-7.35.1
  Attempting uninstall: ml-dtypes
    Found existing installation: ml_dtypes 0.5.4
    Uninstalling ml_dtypes-0.5.4:
      Successfully uninstalled ml_dtypes-0.5.4
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.20.0
    Uninstalling tensorboard-2.20.0:
      Successfully uninstalled tensorboard-2.20.0
ERROR: 

Looking in indexes: https://download.pytorch.org/whl/cpu


In [ ]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [ ]:
!pip install tensorflow


In [ ]:
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
import matplotlib.pyplot as plt
import zipfile
import os
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

print("✅ All imports successful!")
print(f"TensorFlow version: {tf.__version__}")

✅ All imports successful!
TensorFlow version: 2.17.1


In [ ]:
batch_size =32
n_epochs = 5
img_rows, img_cols = 224, 224
input_shape = (img_rows, img_cols, 3)

In [ ]:
import urllib.request
import tarfile
import os

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ZjXM4RKxlBK9__ZjHBLl5A/aircraft-damage-dataset-v1.tar"
tar_filename = "aircraft_damage_dataset_v1.tar"

print("Downloading dataset...")
urllib.request.urlretrieve(url, tar_filename)

print("Extracting dataset...")
with tarfile.open(tar_filename, "r") as tar:
    tar.extractall()

os.remove(tar_filename)

print("Dataset downloaded and extracted successfully!")

Extracting dataset...
Dataset downloaded and extracted successfully!


/tmp/ipykernel_4203/3645846778.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


In [ ]:


extract_path = "aircraft_damage_dataset_v1"
train_dir = os.path.join(extract_path, 'train')
test_dir = os.path.join(extract_path, 'test')
valid_dir = os.path.join(extract_path, 'valid')

In [ ]:
!pip install --upgrade ml_dtypes
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_datagen = ImageDataGenerator(rescale=1./255)
valid_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_rows, img_cols),
    batch_size=batch_size,
    seed=seed_value,
    class_mode='binary',
    shuffle=True
)

Found 300 images belonging to 2 classes.


In [ ]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

model = Sequential()
model.add(base_model)
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))

for layer in base_model.layers:
    layer.trainable = False

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
valid_generator = valid_datagen.flow_from_directory(
    valid_dir,
    target_size=(img_rows, img_cols),
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_rows, img_cols),
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False
)

Found 96 images belonging to 2 classes.
Found 50 images belonging to 2 classes.


In [ ]:
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

history = model.fit(train_generator,
                    epochs=5,
                    validation_data=valid_generator)

Epoch 1/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 293s 29s/step - accuracy: 0.8400 - loss: 0.3235 - val_accuracy: 0.7708 - val_loss: 0.4889
Epoch 2/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 272s 28s/step - accuracy: 0.9000 - loss: 0.2426 - val_accuracy: 0.7604 - val_loss: 0.5233
Epoch 3/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 261s 26s/step - accuracy: 0.9367 - loss: 0.1978 - val_accuracy: 0.7604 - val_loss: 0.4573
Epoch 4/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 277s 29s/step - accuracy: 0.9600 - loss: 0.1517 - val_accuracy: 0.7188 - val_loss: 0.4359
Epoch 5/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 253s 27s/step - accuracy: 0.9867 - loss: 0.1012 - val_accuracy: 0.7292 - val_loss: 0.4375


In [ ]:
def test_generator():
    test_generator = test_datagen.flow_from_directory(
    directory=test_dir,
    class_mode='binary',
    seed=seed_value,
    batch_size=batch_size,
    shuffle=False,
    target_size=(img_rows, img_cols)
)

In [ ]:
base_model = VGG16(weights='imagenet', include_top=False , input_shape=(img_rows, img_cols,3))


In [ ]:
output = base_model.layers[-1].output
output = keras.layers.Flatten()(output)
base_model = Model(base_model.input, output)

for layer in base_model.layers:
    layer.trainable = False

In [ ]:
model = Sequential()
model.add(base_model)
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_generator,
    epochs=n_epochs,
    validation_data=valid_generator
)

Epoch 1/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 259s 26s/step - accuracy: 0.5733 - loss: 0.7031 - val_accuracy: 0.6979 - val_loss: 0.5886
Epoch 2/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 271s 28s/step - accuracy: 0.6700 - loss: 0.6052 - val_accuracy: 0.6562 - val_loss: 0.6030
Epoch 3/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 248s 25s/step - accuracy: 0.7400 - loss: 0.5026 - val_accuracy: 0.5938 - val_loss: 0.8013
Epoch 4/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 245s 25s/step - accuracy: 0.7733 - loss: 0.4947 - val_accuracy: 0.6979 - val_loss: 0.5086
Epoch 5/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 244s 25s/step - accuracy: 0.8567 - loss: 0.3567 - val_accuracy: 0.7188 - val_loss: 0.4667


In [ ]:
train_history = model.history.history

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

def describe_aircraft_damage(image_path):
    img = load_img(image_path, target_size=(224, 224))
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array, verbose=0)
    damage_type = "dent" if prediction[0][0] > 0.5 else "crack"
    confidence = prediction[0][0] if damage_type == "dent" else 1 - prediction[0][0]

    image = Image.open(image_path).convert("RGB")
    prompt = f"An aircraft with a visible {damage_type} on its surface."

    inputs = processor(image, text=prompt, return_tensors="pt")
    output = blip_model.generate(**inputs, max_length=50)
    caption = processor.decode(output[0], skip_special_tokens=True)

    return {
        'damage_type': damage_type,
        'confidence': confidence,
        'description': caption,
        'full_report': f"🔍 DAMAGE DETECTED: {damage_type.upper()} ({confidence:.2%})\n📝 {caption}"
    }

test_image = "aircraft_damage_dataset_v1/test/dent/149_22_JPG_jpg.rf.4899cbb6f4aad9588fa3811bb886c34d.jpg"
result = describe_aircraft_damage(test_image)
print(result['full_report'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

🔍 DAMAGE DETECTED: DENT (95.18%)
📝 an aircraft with a visible dent on its surface., japan
